# RE_105 · Els estius de Barcelona, cada cop més calorosos?

**Optativa d'Aprenentatge automàtic · Bloc 1 — Dades · Dificultat 3**

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Fas pràctiques al Servei Meteorològic de Catalunya. Una televisió prepara un reportatge sobre la
calor a Barcelona i us demana dades de l'Observatori Fabra, l'estació del Tibidabo que fa més de
cent anys que mesura el temps de la ciutat.

**La pregunta:** des del 2016, els estius a Barcelona són cada cop més calorosos? Quins han estat
els dies més extrems, i quin any ha estat el més estrany?

L'enunciat complet, amb el diccionari de columnes, és a [RE_105](../RE_105_meteo_fabra.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FITXER = "xema_fabra_diari.csv"
URL = "https://raw.githubusercontent.com/ZenidX/Machine-Learning/main/Machine%20Learning/04_reptes/dades/" + FITXER
ruta = URL
for candidata in (os.path.join("dades", FITXER), os.path.join("..", "dades", FITXER)):
    if os.path.exists(candidata):
        ruta = candidata
df = pd.read_csv(ruta)
print(ruta, df.shape)

## Nivell 1 · Explorar

### 1.1 · Forma i variables

Quantes files i columnes hi ha? Quants dies diferents? Quantes variables diferents hi ha? Compta-les de dues maneres: pel nom (`nom_variable`) i pel codi (`codi_variable`).

In [ ]:
print(df.shape)
print("dies:", df["data_lectura"].nunique())
print("noms:", df["nom_variable"].nunique(), "· codis:", df["codi_variable"].nunique())

### 1.2 · Variables i unitats

Fes una taula amb una fila per variable: codi, nom i unitat. Ordena-la per codi i mira-la sencera. Per què no surt el mateix nombre de noms que de codis?

In [ ]:
pd.set_option("display.max_rows", 100)
vars_ = df.drop_duplicates(subset=["codi_variable", "nom_variable", "unitat"]).sort_values("codi_variable")
print(len(vars_))
vars_[["codi_variable", "nom_variable", "unitat"]]

**Comentari.** Hi ha 52 noms per a 30 codis: 22 variables tenen una segona versió del nom amb caràcters mal codificats (`�`), i la unitat `°C` també surt trencada. Són 66 lectures de 3 dies (27-29 de juny del 2025, plena onada de calor). A més, algunes variables (vent a 2 m, vent vectorial) només existeixen una part del període.

### 1.3 · Buits i estats

Quants buits hi ha a cada columna? A quines variables té valor `hora_tu`? Quins valors té `estat`, comptant-hi els buits?

In [ ]:
print(df.isna().sum())
amb_hora = df.groupby("codi_variable")["hora_tu"].count()
print(amb_hora[amb_hora > 0])
print(df["estat"].value_counts(dropna=False))
print(df[df["estat"] == "No representatiu"].groupby("codi_variable").size())

## Nivell 2 · Respondre

### 2.1 · Una fila per dia

Passa la taula a format ample: una fila per dia (amb la data com a data de veritat) i una columna per variable, amb el valor. Decideix quina columna fas servir per fer les columnes noves i explica-ho en una línia. Decideix també què fas amb els valors `No representatiu`.

In [ ]:
df["data"] = pd.to_datetime(df["data_lectura"])
w = df.pivot(index="data", columns="codi_variable", values="valor").sort_index()
print(w.shape, df["data"].nunique())
print("cel·les plenes:", w.notna().sum().sum(), "· files llargues:", len(df))
# noms bons de cada codi, per llegir millor
noms = df[~df["nom_variable"].str.contains("\ufffd")].drop_duplicates("codi_variable").set_index("codi_variable")["nom_variable"]
TMIT, TMAX, TMIN, PLUJA = 1000, 1001, 1002, 1300
print(noms[[TMIT, TMAX, TMIN, PLUJA]])
# el que passa si es fa amb el nom
w_nom = df.pivot(index="data", columns="nom_variable", values="valor")
print("columnes amb el nom:", w_nom.shape[1], "· amb el codi:", w.shape[1])

**Comentari.** Pivotar per `codi_variable` és la decisió bona. Amb `nom_variable` surten 52 columnes i els tres dies de l'onada de calor del juny del 2025 queden en columnes "fantasma": la temperatura màxima d'aquells dies desapareix de la columna bona. Els `No representatiu` són de vent i evapotranspiració: no afecten les temperatures.

### 2.2 · Els dies extrems

Quins han estat els cinc dies amb la **temperatura màxima** més alta des del 2016? I els cinc amb més **precipitació acumulada diària**?

In [ ]:
print(w[TMAX].nlargest(5))
print(w[PLUJA].nlargest(5))

### 2.3 · Mapa de calor

Calcula la **temperatura mitjana diària** mitjana de cada mes de cada any: una taula amb els anys a les files i els mesos a les columnes. Dibuixa-la com a mapa de calor, amb els anys i els mesos a les etiquetes.

In [ ]:
taula = w[TMIT].groupby([w.index.year, w.index.month]).mean().unstack()
print(taula.shape)
print(taula.isna().sum(axis=1).to_string())
plt.figure(figsize=(9, 6))
plt.imshow(taula, cmap="coolwarm", aspect="auto")
plt.yticks(range(len(taula)), taula.index)
plt.xticks(range(12), range(1, 13))
plt.colorbar(label="°C")
plt.title("Temperatura mitjana mensual · Observatori Fabra")
plt.show()

### 2.4 · Estius més calorosos?

Per a cada any, calcula (a) la temperatura mitjana de juny, juliol i agost, i (b) el nombre de dies amb màxima de 30 °C o més. Fes-ne un gràfic. Hi veus una tendència?

In [ ]:
estiu = w[w.index.month.isin([6, 7, 8])]
t_estiu = estiu[TMIT].groupby(estiu.index.year).mean()
dies30 = (w[TMAX] >= 30).groupby(w.index.year).sum()
resum = pd.DataFrame({"T mitjana estiu": t_estiu.round(2), "dies >= 30": dies30})
print(resum)
resum.plot(subplots=True, marker="o", figsize=(9, 6), title="Estius a l'Observatori Fabra")
plt.show()
# amb la taula feta pel nom, l'any 2025 perd els dies de l'onada de calor
tmax_nom = w_nom["Temperatura màxima diària + hora"]
print("2025 pel nom:", int((tmax_nom[tmax_nom.index.year == 2025] >= 30).sum()), "· pel codi:", dies30[2025])

### 2.5 · L'any més estrany (NumPy)

Amb la taula de 2.3, agafa només els anys complets i passa-la a un array `T` (un any per fila, 12
mesos per columna).

1. Calcula l'**any mitjà**: la mitjana de cada mes sobre tots els anys.
2. Calcula la distància euclidiana de cada any a l'any mitjà.
3. Quin any s'allunya més del normal? I quin s'hi assembla més?
4. Per a l'any més estrany, mira mes a mes la diferència amb l'any mitjà: on és, l'anomalia?

In [ ]:
completa = taula.dropna()
T = completa.to_numpy()
mitja = T.mean(axis=0)
print(T.shape, mitja.shape)
d = pd.Series(np.linalg.norm(T - mitja, axis=1), index=completa.index).round(2)
print(d.sort_values())
pitjor = d.idxmax()
print(pitjor, (completa.loc[pitjor].to_numpy() - mitja).round(1))

## Nivell 3 · Més enllà

### 3.1 · Dies que falten

Per a cada variable, quants dies del període no tenen valor? Quines variables són completes i quines no? N'hi ha alguna que només existeixi una part del període?

In [ ]:
falten = w.isna().sum()
print(pd.DataFrame({"falten": falten, "primer": w.apply(lambda c: c.first_valid_index().date()), "nom": noms}).sort_values("falten").to_string())

### 3.2 · Nits tropicals

Una nit tropical és la que no baixa de 20 °C. Quantes n'hi ha cada any a l'Observatori Fabra? Compara-ho amb el que has vist a 2.4.

In [ ]:
print((w[TMIN] >= 20).groupby(w.index.year).sum())

## Conclusió de referència

Els estius de l'Observatori Fabra s'escalfen, però no en línia recta: la temperatura mitjana de
juny-agost passa de 22,6 °C el 2016 a 25,4 °C el 2022, baixa a 23,9 i 23,4 el 2023 i el 2024, i
torna a pujar a 25,0 (2025) i 26,0 (2026). Els dies amb màxima de 30 °C o més passen de 23 (2016) a
51 (2025) i 92 (2026). El dia més calorós és el 8-jul-2026 (40,7 °C) i el més plujós, el
15-nov-2018 (139,9 mm). Mes a mes, l'any més allunyat de l'any mitjà és el 2022 (distància 5,44),
amb un maig 2,9 °C per sobre del normal i la segona meitat de l'any entre 1,6 i 2,1 °C més calenta.
Parany: si es pivota pel nom de la variable, els dies 27-29 de juny del 2025 (nom mal codificat)
queden en columnes a part i el 2025 perd 3 dies de calor (48 en lloc de 51).